# NumPy — Example 02: Operations, Broadcasting, and Linear Algebra

> 📘 **Instructor Curriculum** — M03, Python for Data Science (NumPy and Pandas)

**What this notebook is:** the second NumPy pass. [`example_01.ipynb`](./example_01.ipynb)
was about *making* arrays and *reading* them — indexing, slicing, sorting, appending.
This notebook is about *computing with* them: maths on whole arrays, broadcasting,
ready-made array builders, statistics, and linear algebra.

**How to read it:** each Markdown section explains every code cell that follows it, up
to the next section. Most cells print four lines. The useful question at each line is
*what changed from the line above?*

Versions used here: **numpy 2.4.6**, Python 3.14 in `pizza_env`.

---

## The one idea behind this whole notebook

You stop writing loops. You write one expression, and NumPy applies it to every element.

```text
   Python list                        NumPy array
   -----------                        -----------
   out = []                           out = a + b
   for i in range(len(a)):
       out.append(a[i] + b[i])        one line, no loop
```

This is called **vectorization**. The loop still happens — but it happens inside
compiled C code, over one packed block of memory, instead of inside the Python
interpreter.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> You already know this trade-off. Fetching 10,000 rows one `findById` at a time is slow
> because of the per-call overhead, not the work. One batch query is fast. `a + b` is the
> batch query; the Python `for` loop is the N+1 problem.

**Sections 1-4 recap `example_01`** (creating arrays, `ndim`, `dtype`) and add `shape`
and `reshape`. The genuinely new material starts at **section 5, arithmetic**.


---

## 1. Import

`np` is the universal alias. Every NumPy tutorial, book, and error message on the
internet assumes it, so never rename it.

Everything in this notebook hangs off this one import.


In [1]:
import numpy as np

---

## 2. Creating arrays — what `np.array` accepts

`np.array()` takes a Python sequence and copies it into a packed, single-type block of
memory.

| Cell line | Input | Result | Note |
|---|---|---|---|
| `arr1` | list `[1,2,3]` | `[1 2 3]` | the normal case |
| `arr2` | tuple `(4,5,6)` | `[4 5 6]` | tuple works too — the array does not remember which you used |
| `arr3` | list of lists | `[[1 2] [3 4]]` | nesting becomes a second dimension |
| `arr4` | list of booleans | `[True False True]` | `dtype` becomes `bool` |

Two things to notice in the printed output:

1. **No commas.** `[1 2 3]` is a NumPy array. `[1, 2, 3]` is a Python list. That is the
   fastest way to tell them apart when you are reading someone else's output.
2. **The nested list keeps its shape.** The brackets you type are the dimensions you get.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> A Python list is like `ArrayList<Object>` — every slot is a pointer to a boxed object
> living somewhere else in memory. A NumPy array is like `int[]` — the values sit side by
> side in one block. That layout is the whole reason NumPy is fast, and it is also why an
> array can hold only one type.

**Takeaway:** `np.array` accepts any nested sequence, but the result is always one type
in one block of memory.


In [3]:
arr1 = np.array([1,2,3])
arr2 = np.array((4,5,6))
arr3 = np.array([[1,2],[3,4]])
arr4 = np.array([True,False,True])
print(arr1,arr2,arr3,arr4)

[1 2 3] [4 5 6] [[1 2]
 [3 4]] [ True False  True]


---

## 3. `ndim` — how many dimensions?

`ndim` counts the dimensions. The quick way to read it: **count the opening brackets on
the left before the first number.**

```text
np.array([1,2,3])              ->  [1        ->  ndim 1   a vector
np.array([[1,2],[3,4]])        ->  [[1       ->  ndim 2   a matrix / table
np.array([[[1],[2]],...])      ->  [[[1      ->  ndim 3   a stack of tables
np.array([[[[1]]]])            ->  [[[[1     ->  ndim 4   rarely typed by hand
```

Where each one shows up in real work:

| ndim | Name | Real example |
|---|---|---|
| 1 | vector | one row of features, one time series, one column of prices |
| 2 | matrix | a dataset: **rows are samples, columns are features** |
| 3 | 3-D array | one colour image: height x width x colour channel |
| 4 | 4-D array | a batch of images: batch x height x width x channel |

`ndim` 4 looks pointless when you type it by hand. It is what every deep-learning batch
actually is.

**Takeaway:** `ndim` is bracket depth. In real code it tells you what one element *means*
— a value, a row, or a whole table.


In [4]:
print(np.array([1,2,3]).ndim)
print(np.array([[1,2],[3,4]]).ndim)
print(np.array([[[1],[2]],[[3],[4]]]).ndim)
print(np.array([[[[1]]]]).ndim)

1
2
3
4


---

## 4. `shape` and `reshape` — the same data, a different grid

`shape` is a **tuple giving the length of each dimension**. `a = np.arange(6)` gives
6 values and `a.shape` is `(6,)`. The trailing comma is Python's way of writing a
one-item tuple — it is not a typo, and it is not a second empty dimension.

`reshape` lays those same 6 values out on a different grid:

```text
        a = [0 1 2 3 4 5]        shape (6,)      6 values

        reshape(2,3)             reshape(3,2)            reshape(1,6)
        [[0 1 2]                 [[0 1]                  [[0 1 2 3 4 5]]
         [3 4 5]]                 [2 3]
                                  [4 5]]
        2 x 3 = 6  OK            3 x 2 = 6  OK           1 x 6 = 6  OK
```

### The one rule

**The product of the new shape must equal the number of elements.** That is why the last
line is commented out:

```python
a.reshape(1,4)
# ValueError: cannot reshape array of size 6 into shape (1,4)
```

`1 x 4 = 4`, but the array has 6 values. NumPy will not invent two values and will not
drop two. The error saved under that cell is real — it came from running the line before
it was commented out. Keep it; a remembered error message is worth more than a rule.

### Two things worth knowing now

| Point | Detail |
|---|---|
| `reshape` does **not** change `a` | It returns a new *view*. `a` is still `(6,)` after all three calls — which is why the second and third `reshape` calls still work. |
| A view **shares memory** | Verified: `b = a.reshape(2,3)`, then `b[0,0] = 99`, and `a[0]` is now `99`. No copy was made. Use `.copy()` when you need independence. |
| `-1` means "you work it out" | `a.reshape(-1, 2)` gives the same 3x2 result. Useful when the row count depends on data you have not counted. |

**Takeaway:** `shape` is the grid, `reshape` redraws the grid, and the number of values
never changes. When a NumPy error confuses you, print `.shape` first — most NumPy bugs
are shape bugs.


In [ ]:
a = np.arange(6)
print(a.shape)
print(a.reshape(2,3))
print(a.reshape(3,2))
print(a.reshape(1,6))
# print(a.reshape(1,4)) Error here 


(6,)
[[0 1 2]
 [3 4 5]]
[[0 1]
 [2 3]
 [4 5]]
[[0 1 2 3 4 5]]


ValueError: cannot reshape array of size 6 into shape (1,4)

---

## 5. `dtype` — the one type the array holds

Every array has exactly one type, and NumPy picks it from the data unless you say
otherwise.

| Cell line | Input | `dtype` | Why |
|---|---|---|---|
| 1 | `[1,2,3]` | `int64` | whole numbers, 8 bytes each |
| 2 | `[1.1,2.2]` | `float64` | one decimal is enough to make the whole array float |
| 3 | `[1,2,3], dtype=float` | `float64` | you asked for it explicitly |
| 4 | `['ABVFJDDH','BCJ']` | `<U8` | strings |

### Reading `<U8`

```text
   <     U      8
   |     |      |
   |     |      +-- 8 characters: the length of the LONGEST string in the array
   |     +--------- Unicode text
   +--------------- little-endian byte order
```

This is the string trap. `'BCJ'` is 3 characters, but it is stored in an 8-character
slot, because a packed block needs every slot the same size. Verified: `itemsize` is
**32 bytes** per element — 4 bytes per character x 8.

And the consequence people hit for real: assigning a longer string into a `<U8` array
**silently truncates it**. When you work with text, this is the moment to reach for
Pandas instead.

**Takeaway:** `dtype` is decided once, from the data or from you. Mixed input gets
promoted to the type that can hold everything — `int` loses to `float`, and everything
loses to text.


In [6]:
print(np.array([1,2,3]).dtype)
print(np.array([1.1,2.2]).dtype)
print(np.array([1,2,3],dtype=float).dtype)
print(np.array(['ABVFJDDH','BCJ']).dtype)

int64
float64
float64
<U8


---

## 6. Arithmetic — one operator, every element

This is the payoff for the packed memory layout.

```python
a = np.array([1,2,3])
b = np.array([4,5,6])
a + b        ->  [5 7 9]
```

Position by position: `1+4`, `2+5`, `3+6`. The same applies to `-`, `*`, and `/`.

| Operator | Result | Read it as |
|---|---|---|
| `a + b` | `[5 7 9]` | element-wise sum |
| `a - b` | `[-3 -3 -3]` | element-wise difference |
| `a * b` | `[4 10 18]` | element-wise product — **not** matrix multiplication |
| `a / b` | `[0.25 0.4 0.5]` | element-wise division, always `float64` |

### The two traps

**1. `*` is not matrix multiplication.** `a * b` multiplies matching positions. The
matrix product is `@` or `np.matmul`, in section 15. In maths notation both are written
with a dot, so this catches almost everyone once.

**2. `/` always returns floats.** `int64 / int64` gives `float64`, even when the division
is exact. NumPy chooses the result type by the operation, not by the values.

**3. Shapes must match** — or be broadcastable, which is the next section.

**Takeaway:** arithmetic on arrays is position-by-position, and it replaces the loop you
would have written in plain Python.


In [7]:
a=np.array([1,2,3])
b=np.array([4,5,6])
print(a+b)
print(a-b)
print(a*b)
print(a/b)

[5 7 9]
[-3 -3 -3]
[ 4 10 18]
[0.25 0.4  0.5 ]


---

## 7. Comparisons — the boolean mask

A comparison is arithmetic too. It runs on every element and returns an array of `True`
and `False`, the same shape as the input.

```text
   a        =  [10   20    30  ]
   a > 15   ->  [False True True]
```

| Cell line | Meaning | Result |
|---|---|---|
| `a > 15` | which values are above 15? | `[False True True]` |
| `a == 20` | which equal 20? | `[False True False]` |
| `a != 20` | which are not 20? | `[True False True]` |
| `a <= 20` | which are 20 or less? | `[True True False]` |

That result array is called a **mask**, and it is the point of the whole exercise. A mask
is not usually printed — it is fed straight back into the array:

```python
a[a > 15]        # -> [20 30]     keep the values that passed
a[a > 15] = 0    #                overwrite only those values
(a > 15).sum()   # -> 2           True counts as 1, so sum = how many passed
```

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> `a[a > 15]` is a `WHERE` clause. You describe the condition and NumPy applies it to
> the whole column at once — no cursor, no row loop. Pandas' `df[df.age > 30]` is exactly
> this same mechanism, one layer up.

### The one real trap

Use `&`, `|`, `~` for combining conditions — **not** `and`, `or`, `not`:

```python
a[(a > 15) & (a < 30)]     # correct
a[a > 15 and a < 30]       # ValueError: truth value of an array is ambiguous
```

Python's `and` wants one true-or-false answer. An array of three booleans cannot give
one. Also keep the brackets: `&` binds tighter than `>`.

**Takeaway:** a comparison gives you a mask, and a mask is how you filter, count, and
update parts of an array without a loop.


In [8]:
a=np.array([10,20,30])
print(a>15)
print(a==20)
print(a!=20)
print(a<=20)

[False  True  True]
[False  True False]
[ True False  True]
[ True  True False]


---

## 8. Broadcasting — when shapes do not match

Section 6 needed matching shapes. Broadcasting is the set of rules that lets NumPy work
with mismatched shapes by **stretching the smaller one**, without copying it.

| Cell line | Shapes | Result | What was stretched |
|---|---|---|---|
| `a + 10` | `(3,)` + scalar | `[11 12 13]` | `10` acts as `[10 10 10]` |
| `a * 5` | `(3,)` + scalar | `[5 10 15]` | same idea |
| `a + [10,20,30]` | `(3,)` + `(3,)` | `[11 22 33]` | nothing — the list is just converted to an array |
| `[[1],[2],[3]] + [10,20]` | `(3,1)` + `(2,)` | a **3x2** matrix | both were stretched |

### The last one, drawn out

This is the cell worth understanding. A column of 3 plus a row of 2 gives a 3x2 grid —
every pairing.

```text
   column (3,1)          row (2,)              result (3,2)
                         [10  20]
      [1]                                      [11  21]      1+10  1+20
      [2]        +                    =        [12  22]      2+10  2+20
      [3]                                      [13  23]      3+10  3+20

   the column is copied across 2 columns
   the row    is copied down   3 rows      (conceptually — no memory is actually copied)
```

### The rule

Line the shapes up **from the right**. Each pair of dimensions must be either equal, or
one of them must be 1.

```text
   (3,1)                (3,)
   (  2)                (4,)
   -----                -----
   3 vs nothing -> 3    3 vs 4 -> neither equal nor 1
   1 vs 2       -> 2                 ValueError: operands could not be broadcast
   = (3,2)  OK
```

### Where you will actually use this

Centring a dataset. `X` is `(1000, 5)` — 1000 samples, 5 features. `X.mean(axis=0)` is
`(5,)` — one mean per feature.

```python
X_centered = X - X.mean(axis=0)      # (1000,5) - (5,) -> (1000,5)
```

One row of 5 means is applied to all 1000 rows. That is feature scaling, and it is one
line because of broadcasting.

**Takeaway:** broadcasting stretches size-1 (or missing) dimensions so shapes fit. It is
what makes "subtract the mean from every row" a single expression — but it will also
silently build a huge matrix if you mix up a row and a column, so check the result shape.


In [9]:
a=np.array([1,2,3])
print(a+10)
print(a*5)
print(a+[10,20,30])
print(np.array([[1],[2],[3]])+np.array([10,20]))

[11 12 13]
[ 5 10 15]
[11 22 33]
[[11 21]
 [12 22]
 [13 23]]


---

## 9. Ready-made arrays — build without data

Often you need an array of the right *shape* before you have any values. These four
builders take the shape as a tuple.

| Call | Result | Typical use |
|---|---|---|
| `np.zeros((2,2))` | all `0.` | an empty accumulator you will fill in |
| `np.ones((2,2))` | all `1.` | a starting weight, or a mask you multiply by |
| `np.full((2,2), 9)` | all `9` | any other constant, including placeholders |
| `np.eye(3)` | 1s on the diagonal | the **identity matrix** |

### Two details in the output

**The dots.** `zeros` and `ones` print `0.` and `1.` — they default to `float64`. `full`
printed `9` with no dot, because it took its type from the fill value you passed. If you
need integer zeros, ask: `np.zeros((2,2), dtype=int)`.

**`np.eye(3)` takes one number, not a tuple.** An identity matrix is always square, so
one number is enough.

### Why the identity matrix matters

It is the `1` of matrix multiplication: `A @ I == A`, for any `A`. That is what makes it
the check for section 18 — if `inv(A)` is correct, then `A @ inv(A)` gives back the
identity.

**Takeaway:** these four give you a correctly shaped array before you have values. Watch
the dtype — the default is float.


In [10]:
print(np.zeros((2,2)))
print(np.ones((2,2)))
print(np.full((2,2),9))
print(np.eye(3))

[[0. 0.]
 [0. 0.]]
[[1. 1.]
 [1. 1.]]
[[9 9]
 [9 9]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


---

## 10. `arange` and `linspace` — building sequences

Both make evenly spaced numbers. They differ in **which part you control**.

| Call | Result | You control |
|---|---|---|
| `np.arange(1,10)` | `[1 2 ... 9]` | the **step** (default 1) |
| `np.arange(1,10,2)` | `[1 3 5 7 9]` | step of 2 |
| `np.linspace(0,1,5)` | `[0. 0.25 0.5 0.75 1.]` | the **count** (5 values) |
| `np.linspace(10,20,4)` | `[10. 13.33 16.67 20.]` | 4 values, spacing worked out for you |

### The endpoint rule — they disagree, and it matters

```text
   np.arange(1,10)      ->  1 ... 9      stop is EXCLUDED   (like range())
   np.linspace(0,1,5)   ->  0.0 ... 1.0  stop is INCLUDED   (both ends kept)
```

`arange` follows Python's `range`. `linspace` is built for plotting and sampling, where
you want both ends of the interval.

### Choosing between them

```text
   Do I care about the gap between values?   ->  arange
   Do I care how many values I get?          ->  linspace
```

### The floating-point trap

Never use `arange` with a decimal step when the count matters:

```python
np.arange(0.1, 0.4, 0.1)     # -> [0.1 0.2 0.3 0.4]   FOUR values, not three
```

Verified. `0.1` cannot be stored exactly in binary, the accumulated error lands just
under `0.4`, and one extra value slips in. `np.linspace(0.1, 0.4, 4)` asks for a count
and cannot do this. **Integers: `arange`. Decimals: `linspace`.**

**Takeaway:** `arange` for a known step, `linspace` for a known count. `arange` excludes
the stop; `linspace` includes it.


In [11]:
print(np.arange(1,10))
print(np.arange(1,10,2))
print(np.linspace(0,1,5))
print(np.linspace(10,20,4))

[1 2 3 4 5 6 7 8 9]
[1 3 5 7 9]
[0.   0.25 0.5  0.75 1.  ]
[10.         13.33333333 16.66666667 20.        ]


---

## 11. Random numbers — four different generators

These four are not variations on one idea. Each draws from a different distribution.

| Call | What it draws | Range | Shape given by |
|---|---|---|---|
| `np.random.randint(1,10,5)` | whole numbers, all equally likely | 1 to **9** — high is excluded | last argument `5` |
| `np.random.rand(3)` | decimals, uniform (flat) | 0.0 to 1.0 | `3` |
| `np.random.randn(3)` | decimals, **normal** (bell curve) | mostly -3 to 3, mean 0 | `3` |
| `np.random.choice([10,20,30],5)` | picks *from your list* | your values | `5` |

Two details in the printed output:

- `randn` produced a **negative** number. It is the only one of the four that can. If
  you need "random data" that looks like a real measurement — noise, heights, errors —
  this is the one.
- `choice` repeated values. It samples **with replacement** by default. Pass
  `replace=False` for a sample without repeats.

### The reproducibility problem

Run the cell twice and you get different numbers. That is correct behaviour, and it is a
problem the moment you want to compare two runs, debug a result, or let someone else
reproduce your notebook.

```python
np.random.seed(42)                 # fixes every np.random.* call afterwards
```

Every ML tutorial you read will do this before splitting data. It is why `random_state=42`
appears everywhere in scikit-learn.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> `np.random.seed` sets **one global generator** — shared, hidden state, exactly the kind
> of thing you would refuse to ship in a Spring service. Modern NumPy gives you an object
> instead:
>
> ```python
> rng = np.random.default_rng(42)
> rng.integers(1, 10, 5)      # note: integers(), not randint()
> rng.random(3)
> rng.normal(size=3)
> ```
>
> Use `np.random.*` to follow along with the course and with tutorials. Use
> `default_rng` in anything you write for real.

**Takeaway:** pick the generator by the distribution you need — uniform, normal, or a
draw from your own values — and set a seed whenever the result has to be repeatable.


In [12]:
print(np.random.randint(1,10,5))
print(np.random.rand(3))
print(np.random.randn(3))
print(np.random.choice([10,20,30],5))

[8 6 2 6 8]
[0.71121896 0.45439005 0.08269947]
[ 0.85181675  0.46243431 -1.0139461 ]
[30 10 30 30 20]


---

## 12. Statistics — describing an array

Seven summaries of `[10, 20, 30, 40]`.

| Call | Result | Meaning |
|---|---|---|
| `np.sum` | `100` | total |
| `np.min` / `np.max` | `10` / `40` | the extremes |
| `np.mean` | `25.0` | the average |
| `np.median` | `25.0` | the middle value after sorting |
| `np.std` | `11.1803...` | typical distance from the mean |
| `np.var` | `125.0` | std squared |

`mean` and `median` agree here because the data is evenly spread. When they disagree, the
gap is telling you about **skew or outliers** — that is the first thing to check in real
data. Change `40` to `4000` and the mean jumps to 1015 while the median moves to 25.
Median is the robust one.

`std` and `var` are the same information: `std = sqrt(var)`, and `sqrt(125) = 11.18`.
Variance is used inside maths; standard deviation is what you report, because it is in
the same unit as the data.

### The trap that will bite you later

**`np.std` divides by `n`. Pandas divides by `n-1`.** Same data, different answer:

```text
   np.std([10,20,30,40])              ->  11.1803...     ddof=0, population
   pd.Series([10,20,30,40]).std()     ->  12.9099...     ddof=1, sample
```

Both verified above. NumPy assumes your array **is** the whole population. Pandas assumes
it is a *sample* of something bigger, and corrects for it. Neither is wrong; they answer
different questions. When two tools report different standard deviations for the same
column, this is why. Force it with `np.std(arr, ddof=1)`.

### `axis` — the argument you will need immediately

On a 2-D array, these functions collapse everything by default. `axis` says which
direction to collapse:

```text
   X = [[1 2]        X.sum()          -> 10     everything
        [3 4]]       X.sum(axis=0)    -> [4 6]  down the rows  -> one result per COLUMN
                     X.sum(axis=1)    -> [3 7]  across columns -> one result per ROW
```

Read `axis=0` as "the row axis disappears". With rows as samples and columns as features,
`axis=0` is per-feature — which is what section 8's `X.mean(axis=0)` was doing.

**Takeaway:** these functions summarise a whole array in one call. Remember two things —
`np.std` is population by default, and `axis` decides which dimension collapses.


In [16]:
arr=np.array([10,20,30,40])
print(np.sum(arr))
print(np.min(arr))
print(np.max(arr))
print(np.mean(arr))
print(np.median(arr))
print(np.std(arr))
print(np.var(arr))


100
10
40
25.0
25.0
11.180339887498949
125.0


---

## 13. `np.info` — looking at how the array is stored

`np.info` prints the array's physical layout. Most of these functions describe values;
this one describes **memory**.

| Field | Value here | Meaning |
|---|---|---|
| `class` | `ndarray` | the NumPy array type |
| `shape` | `(4,)` | 4 values in one dimension |
| `strides` | `(8,)` | move **8 bytes** to reach the next element |
| `itemsize` | `8` | each `int64` takes 8 bytes |
| `contiguous` | `True` | the values sit in one unbroken block |
| `type` | `int64` | the dtype |

`strides` is the interesting one and it explains section 4. Reshaping does not move any
data — NumPy just changes the shape and the stride numbers, and reads the *same* bytes in
a different pattern. That is why `reshape` returns a view and why it is free.

`fortran: True` here only because a 1-D array is both row-major and column-major at once.
On a 2-D array it would be `False`.

### One small thing about the cell itself

```python
print(np.info(arr))     # prints the report, then prints None
np.info(arr)            # better
```

`np.info` prints its report itself and returns `None`, so wrapping it in `print` adds a
stray `None` at the end of the output — visible in this cell. It is a harmless bug, and
worth recognising: **a function that prints is not a function that returns.**

**Takeaway:** `np.info` shows the memory layout. Use it when you want to know whether an
operation copied the data or just re-read it.


In [17]:
print("Information about array")
print(np.info(arr))

Information about array
class:  ndarray
shape:  (4,)
strides:  (8,)
itemsize:  8
aligned:  True
contiguous:  True
fortran:  True
data pointer: 0x85d878580
byteorder:  little
byteswap:  False
type: int64
None


---

## 14. The dot product — the most important operation in ML

```python
np.dot(a, b)    # -> 32
a @ b           # -> 32   identical
```

Multiply matching positions, then add everything up. Two arrays go in, **one number**
comes out:

```text
   a = [1  2  3]
   b = [4  5  6]
        |  |  |
        v  v  v
        4 +10 +18   =  32
```

`@` is the operator form, added to Python for exactly this. Prefer it — `a @ b` reads
better than `np.dot(a, b)` once expressions get longer.

### Why this one operation matters

A dot product is a **weighted sum**, and a weighted sum is what almost every model
computes:

```python
features = np.array([1200, 3, 15])          # size, bedrooms, age
weights  = np.array([  85, 12000, -400])    # learned from data

price = features @ weights                  # 1200*85 + 3*12000 + 15*-400
```

That single expression is linear regression's prediction. It is also one neuron in a
neural network. Change the numbers and the meaning changes; the operation does not.

The other common use is **similarity**. The cosine similarity behind semantic search and
RAG retrieval is a dot product of two normalised vectors.

**Takeaway:** the dot product turns two vectors into one number by multiplying and
adding. Learn to read `@` as "weighted sum" and a lot of ML code becomes readable.


In [18]:
a=np.array([1,2,3])
b=np.array([4,5,6])
print(np.dot(a,b))
print(a@b)

32
32


---

## 15. Matrix multiplication — many dot products at once

Same operator, two dimensions. `np.matmul(a, b)` and `a @ b` give the same result.

```text
   a = [[1 2]        b = [[5 6]        a @ b = [[19 22]
        [3 4]]             [7 8]]               [43 50]]
```

Every output cell is **one dot product**: row `i` of `a` with column `j` of `b`.

```text
   result[0,0] = row0 . col0 = 1*5 + 2*7 = 19
   result[0,1] = row0 . col1 = 1*6 + 2*8 = 22
   result[1,0] = row1 . col0 = 3*5 + 4*7 = 43
   result[1,1] = row1 . col1 = 3*6 + 4*8 = 50
```

### The shape rule

```text
   (m, k) @ (k, n)  ->  (m, n)
        ^     ^
        the inner numbers must match, and they vanish
```

`(2,3) @ (3,4)` gives `(2,4)`. `(2,3) @ (2,3)` is an error. When a matrix error appears,
print both shapes and line them up — the inner pair is nearly always the problem.

### `dot` vs `matmul` vs `*`

| Expression | On 2-D arrays |
|---|---|
| `a * b` | element-wise — position by position, **not** a matrix product |
| `np.dot(a,b)` | matrix product |
| `a @ b` | matrix product |

`np.dot` and `@` agree for 1-D and 2-D (verified). They differ beyond that: for 3-D+
arrays `@` treats the leading dimensions as a *batch* of matrices, which is what you want
for batched data, while `np.dot` does something else. `np.dot(3,4)` also happily returns
`12`, while `3 @ 4` raises an error — `@` refuses scalars on purpose.

**Use `@`.** It is clearer and its stacked behaviour is the one that scales.

### Where it shows up

One row of features `@` weights gives one prediction. A whole **matrix** of 1000 samples
`@` the same weights gives 1000 predictions in one call — no loop over rows. That is a
forward pass.

**Takeaway:** matrix multiplication is a grid of dot products. Watch the inner
dimensions, and use `@`.


In [19]:
a=np.array([[1,2],[3,4]])
b=np.array([[5,6],[7,8]])
print(np.matmul(a,b))
print(a@b)

[[19 22]
 [43 50]]
[[19 22]
 [43 50]]


---

## 16. Sorting and searching

Four calls on `[40, 10, 30, 20]`.

| Call | Result | What it gives back |
|---|---|---|
| `np.sort(arr)` | `[10 20 30 40]` | the **values**, sorted |
| `np.argsort(arr)` | `[1 3 2 0]` | the **positions** that would sort it |
| `np.where(arr==30)` | `(array([2]),)` | positions where the condition is `True` |
| `np.searchsorted(np.sort(arr),25)` | `2` | where 25 *would* go to keep order |

### `argsort` — read the answer carefully

`[1 3 2 0]` is not data. It is instructions: *"take index 1 first, then 3, then 2, then
0."* Follow it: `arr[1]=10`, `arr[3]=20`, `arr[2]=30`, `arr[0]=40` — sorted.

Why bother? Because it sorts *other* things by this array:

```python
order = np.argsort(scores)
names[order]              # names, ordered by score
names[np.argsort(scores)[::-1]][:3]      # top 3
```

Sorting one array by another array is the everyday use. Pandas' `sort_values` is this,
underneath.

### `np.where` returns a tuple

Note the shape of the output: `(array([2]),)` — a **tuple** holding one array, not a bare
array. There is one array per dimension, so a 2-D array gives you `(rows, cols)`. To get
the plain indices from a 1-D array, take `[0]`:

```python
np.where(arr == 30)[0]     # -> array([2])
```

Forgetting the `[0]` is a normal first mistake.

### `searchsorted` needs sorted input

It runs a binary search, so it is only correct on a sorted array — hence the
`np.sort(arr)` inside the call. It answers "at which index does 25 belong?" Answer: `2`,
between 20 and 30. This is how you bucket values into bins or ranges without a loop.

**Takeaway:** `sort` gives values, `argsort` gives positions. Reach for `argsort` when
the array you are sorting by is not the array you want back.


In [20]:
arr=np.array([40,10,30,20])
print(np.sort(arr))
print(np.argsort(arr))
print(np.where(arr==30))
print(np.searchsorted(np.sort(arr),25))

[10 20 30 40]
[1 3 2 0]
(array([2]),)
2


---

## 17. Joining and splitting arrays

> ⚠️ **This cell has no saved output — run it.** The behaviour below is verified in the
> same environment, but the notebook itself does not show it yet.

| Call | Result | Direction |
|---|---|---|
| `np.vstack((a,b))` | `[[1 2] [3 4]]` | **v**ertical — stacks as **rows**, 1-D becomes 2-D |
| `np.hstack((a,b))` | `[1 2 3 4]` | **h**orizontal — joins end to end, stays 1-D |
| `np.split(arr,3)` | `[array([1,2]), array([3,4]), array([5,6])]` | cuts into 3 equal pieces |

```text
   a = [1 2]   b = [3 4]

   vstack           hstack
   [[1 2]           [1 2 3 4]
    [3 4]]

   grows a new dimension        stays flat, gets longer
```

### Two things to notice

**The double brackets.** `np.vstack((a,b))` — the arrays go in as **one tuple argument**,
not as two arguments. `np.vstack(a, b)` is an error. The same applies to `hstack` and
`concatenate`.

**`split` returns a Python list**, not an array — a list of arrays. So `np.split(arr,3)[0]`
is the first piece.

### The `split` rule

`np.split` demands an **equal** division:

```python
np.split(np.array([1,2,3,4,5]), 3)
# ValueError: array split does not result in an equal division
```

Use `np.array_split` when it might not divide evenly — verified, it gives
`[array([1,2]), array([3,4]), array([5])]` and puts the remainder in the last pieces.

This is exactly how a dataset gets cut into batches or folds, and it is why
`array_split` is the safer default: real row counts are rarely divisible by your batch
size.

**Takeaway:** `vstack` adds rows, `hstack` extends the row, and both take a single tuple.
`split` needs an even division; `array_split` does not.


In [ ]:
a=np.array([1,2])
b=np.array([3,4])
print(np.vstack((a,b)))
print(np.hstack((a,b)))
arr=np.array([1,2,3,4,5,6])
print(np.split(arr,3))

---

## 18. Linear algebra — `np.linalg`

Three operations on `A = [[1,2],[3,4]]`. This is the module that connects NumPy to the
maths behind ML.

| Call | Result | Meaning |
|---|---|---|
| `np.linalg.inv(A)` | `[[-2. 1.] [1.5 -0.5]]` | the **inverse** — the matrix that undoes `A` |
| `np.linalg.det(A)` | `-2.0000000000000004` | the **determinant** — a single number describing the matrix |
| `np.linalg.eig(A)` | values `[-0.372, 5.372]` + vectors | the **eigen**decomposition |

### The inverse

`inv(A)` is the matrix version of `1/x`. The check is the identity matrix from section 9:

```python
A @ np.linalg.inv(A)     # -> [[1. 0.] [0. 1.]]     verified
```

If a matrix multiplication is the operation, the inverse is the undo.

### The determinant, and why it is `-2.0000000000000004`

That trailing `...04` is not a NumPy bug — it is binary floating point. The true answer
is `-2`, computed through divisions that cannot be represented exactly. **Never compare
floats with `==`;** use `np.isclose(det, -2)`.

The determinant also tells you whether the inverse exists at all:

```text
   det != 0   ->  invertible
   det == 0   ->  singular, no inverse
```

Verified: `np.linalg.inv([[1,2],[2,4]])` raises `LinAlgError: Singular matrix`. The second
row there is just the first row doubled — it carries no new information. In data terms,
that is a perfectly correlated feature, and it is the same reason multicollinearity breaks
linear regression.

> 💡 **Engineering Extension** — *Recommended Extension - Not part of instructor curriculum.*
>
> To *solve* `Ax = b`, use `np.linalg.solve(A, b)` rather than `inv(A) @ b`. It is faster
> and numerically more stable. Computing an explicit inverse is usually a sign you are
> doing more work than the problem needs.

### Eigenvalues and eigenvectors

An eigenvector is a direction that `A` does not rotate — it only stretches it. The
eigenvalue is the stretch factor.

```text
   A @ v  =  lambda * v
```

Verified for the first pair: `A @ vecs[:,0]` and `vals[0] * vecs[:,0]` both give
`[0.30697009, -0.21062466]`.

**Read the eigenvectors as columns, not rows.** `vecs[:, 0]` goes with `vals[0]`. Taking
`vecs[0]` — the first *row* — is the standard mistake here and it silently gives you
nonsense.

This is the machinery under **PCA**: the eigenvector with the largest eigenvalue is the
direction in which your data varies most, and that becomes the first principal component.

**Takeaway:** `np.linalg` is where the maths lives. `det == 0` means no inverse, floats
never compare exactly, and eigenvectors are columns.


In [21]:
a=np.array([[1,2],[3,4]])
print(np.linalg.inv(a))
print(np.linalg.det(a))
vals,vecs=np.linalg.eig(a)
print(vals)
print(vecs)

[[-2.   1. ]
 [ 1.5 -0.5]]
-2.0000000000000004
[-0.37228132  5.37228132]
[[-0.82456484 -0.41597356]
 [ 0.56576746 -0.90937671]]


---

## Recap — what this notebook added

| Section | You can now |
|---|---|
| 4 | Read `shape` and reshape data without changing its values |
| 6-7 | Do maths and comparisons on whole arrays, and filter with a boolean mask |
| 8 | Explain why `(3,1) + (2,)` gives a 3x2 result |
| 9-10 | Build arrays before you have data, and choose `arange` vs `linspace` |
| 11 | Pick the right random generator, and make a run reproducible |
| 12 | Summarise data, and remember `np.std` is population by default |
| 14-15 | Read `@` as a weighted sum, and check the inner dimensions |
| 16 | Sort one array by another with `argsort` |
| 17-18 | Join, split, invert, and decompose |

### The five gotchas worth memorising

1. `*` is element-wise. `@` is the matrix product.
2. `arange` excludes the stop value; `linspace` includes it — and `arange` with a decimal
   step is unreliable.
3. `np.std` uses `ddof=0`; Pandas uses `ddof=1`.
4. `np.where` returns a **tuple** — take `[0]` for 1-D.
5. Eigenvectors come back as **columns**: `vecs[:, i]` matches `vals[i]`.

### Not covered yet

`axis` used properly across 2-D arrays, `np.concatenate` and `np.stack`, `np.clip`,
`np.unique`, `np.isnan` and missing values, `np.newaxis`, cumulative functions
(`cumsum`, `cumprod`), and views vs copies in depth. Several of these appear in
[`project_01_api_health.ipynb`](./project_01_api_health.ipynb), which is where these
operations get used on one realistic dataset.

**If you remember only one thing:** NumPy replaces loops with expressions on whole
arrays — and the price of that power is that **shape is everything**. When something
breaks, print `.shape` before you print the values.
